# 03 · Make the hallucinated texts and run Qwen over them

Everything here is kept apart from the 25 real texts and their results (`texts_hallucinated/`, `results/run_hallucinated/`).

| texts | how they are made | answer key |
|---|---|---|
| **planted** (`texts_hallucinated/planted/`) | a whole real text with 2–3 nouns/verbs swapped for a hallucination | the `{{original => planted \| type}}` markers in the file (type: entity, relation, invented) |
| **low temperature** (`*_t07`) | Qwen writes a story at temperature 0.7 | hallucinations marked by reading (`generated/marks.csv`) |
| **high temperature** (`*_t10`) | the same request at temperature 1.0 | same |
| **switch** (`*_switch`) | the 0.7 story up to a random switch point, continued at 1.0 | the switch point (`generated/switch_points.csv`) + marks |

Then Qwen reads every text exactly like notebook 01 does (same model, same measurements).
Only the planted words that are nouns or verbs (not names) can show up on the word curves.

**Runtime:** loading ~30 s; each story ~1 minute to write; each text 3–5 minutes to read. Finished texts are skipped.

In [1]:
import os
import sys
import time
import warnings
from pathlib import Path

os.environ.setdefault("HF_HUB_OFFLINE", "1")   # the model is already in the Hugging Face cache
warnings.filterwarnings("ignore")
# make this folder's qwen_config.py and analysis/ importable, wherever Jupyter was started (e.g. the repo root)
QWEN7B_DIR = Path.cwd()
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "src/note_books/Qwen7B/qwen_config.py").exists():
        QWEN7B_DIR = folder / "src/note_books/Qwen7B"
        break
sys.path.insert(0, str(QWEN7B_DIR))

import analysis                                # also makes lca_core (repo root) importable
import numpy as np
import pandas as pd
import torch

from analysis.generate import decode, switch_story, write_story
from analysis.hallucinations import generated_texts, planted_texts
from qwen_config import (GENERATED_DIR, GUESSES_PER_WORD, HALLUCINATED_RUN_DIR, HALLUCINATED_TOKENS_DIR,
                         HALLUCINATED_WORDS_DIR, HIGH_TEMPERATURE, LOAD_IN_4BIT, LOW_TEMPERATURE, MODEL_NAME,
                         RANDOM_SEED, SECOND_READER_MODEL, SECOND_READER_TOKENS_DIR, STORY_PROMPTS, TEXTS_DIR,
                         WHOLE_WORD_GUESSES)
from lca_core import LCAMeter
from src.model.load_model import load_model

for folder in [GENERATED_DIR, HALLUCINATED_WORDS_DIR, HALLUCINATED_TOKENS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

## 1. The planted texts

In [2]:
planted = planted_texts()
edit_rows = []
for name, (original_name, _text, edits) in planted.items():
    for edit in edits:
        edit_rows.append({"text": name, "clean original": original_name, **edit})
pd.DataFrame(edit_rows)

,text,clean original,original,planted,type,char
0,creative_gut_grimm_planted,creative_gut_grimm,died,married,relation,500
1,creative_gut_grimm_planted,creative_gut_grimm,Hans,Peter,entity,1244
2,creative_gut_grimm_planted,creative_gut_grimm,beer,milk,entity,1639
3,creative_gut_grimm_planted,creative_gut_grimm,masons,dragons,invented,2205
4,creative_gut_oz_planted,creative_gut_oz,Oz,Glinda,entity,806
5,creative_gut_oz_planted,creative_gut_oz,elephant,mouse,entity,1462
6,creative_gut_oz_planted,creative_gut_oz,tin,flesh,entity,2176
7,creative_gut_oz_planted,creative_gut_oz,kill,marry,relation,2553
8,creative_gut_sherlock_planted,creative_gut_sherlock,Holmes,Watson,entity,186
9,creative_gut_sherlock_planted,creative_gut_sherlock,head,foot,entity,520


## 2. Load the model

In [3]:
started = time.time()
tokenizer, model = load_model(MODEL_NAME, load_in_4bit=LOAD_IN_4BIT)
meter = LCAMeter(model=model, tokenizer=tokenizer)
print(f"loaded in {time.time() - started:.0f} s")

system check:
Python path: C:\Users\yoav7\miniconda3\python.exe
Torch version: 2.6.0+cu124
GPU: NVIDIA GeForce RTX 4060
PyTorch: 2.6.0+cu124
CUDA: 12.4
CUDA available: True
Capability: (8, 9)
this is include the following functions torch , torch.nn.functional as F, transformers.AutoTokenizer, transformers.AutoModelForCausalLM


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

loaded in 28 s


## 3. Write the stories
For each request: the 0.7 story, the switch story (0.7 up to a random switch point, then 1.0), and the 1.0 story.
The random draws use `RANDOM_SEED`. Stories already written are kept (their hallucinations are marked by hand
in `generated/marks.csv`); delete `generated/switch_points.csv` to write them again.

In [4]:
rng = np.random.default_rng(RANDOM_SEED)
switch_rows = []
already_written = (GENERATED_DIR / "switch_points.csv").exists()
for story_number, (story_key, prompt) in enumerate(STORY_PROMPTS.items()):
    if already_written:
        break
    seed = RANDOM_SEED + story_number
    low_name = f"creative_gen_{story_key}_t{round(LOW_TEMPERATURE * 10):02d}"
    high_name = f"creative_gen_{story_key}_t{round(HIGH_TEMPERATURE * 10):02d}"
    switch_name = f"creative_gen_{story_key}_switch"
    started = time.time()

    low_ids = write_story(model, tokenizer, prompt, LOW_TEMPERATURE, seed)
    (GENERATED_DIR / f"{low_name}.txt").write_text(decode(tokenizer, low_ids), encoding="utf-8")

    switch_ids, switch_token = switch_story(model, tokenizer, prompt, low_ids, seed, HIGH_TEMPERATURE, rng)
    (GENERATED_DIR / f"{switch_name}.txt").write_text(decode(tokenizer, switch_ids), encoding="utf-8")
    switch_rows.append({"text": switch_name, "clean original": low_name, "switch token": switch_token,
                        "switch char": len(decode(tokenizer, low_ids[:switch_token])), "story tokens": len(switch_ids)})

    high_ids = write_story(model, tokenizer, prompt, HIGH_TEMPERATURE, seed)
    (GENERATED_DIR / f"{high_name}.txt").write_text(decode(tokenizer, high_ids), encoding="utf-8")
    print(f"{story_key:10s} {len(low_ids)} / {len(switch_ids)} / {len(high_ids)} tokens, "
          f"switch at token {switch_token}, {time.time() - started:.0f} s", flush=True)

if already_written:
    switch_points = pd.read_csv(GENERATED_DIR / "switch_points.csv")
else:
    switch_points = pd.DataFrame(switch_rows)
    switch_points.to_csv(GENERATED_DIR / "switch_points.csv", index=False)
switch_points

,text,clean original,switch token,switch char,story tokens
0,creative_gen_magic_switch,creative_gen_magic_t07,557,2536,806
1,creative_gen_fairy_switch,creative_gen_fairy_t07,465,2081,818
2,creative_gen_detective_switch,creative_gen_detective_t07,506,2269,1000


## 4. Qwen reads every text
Same measurements as notebook 01. Texts with saved results are skipped.

In [5]:
texts_to_read = {}
for name, (_original, text, _edits) in planted.items():
    texts_to_read[name] = text
texts_to_read.update(generated_texts())

log_path = HALLUCINATED_RUN_DIR / "run_log.csv"
if log_path.exists():
    run_log = pd.read_csv(log_path)
else:
    run_log = pd.DataFrame(columns=["text", "measured words", "tokens", "seconds"])

for name, text in texts_to_read.items():
    words_path = HALLUCINATED_WORDS_DIR / f"{name}.csv"
    tokens_path = HALLUCINATED_TOKENS_DIR / f"{name}.csv"
    if words_path.exists() and tokens_path.exists():
        continue
    started = time.time()
    words = meter.run(text, k=GUESSES_PER_WORD, complete_words=WHOLE_WORD_GUESSES)
    words.to_csv(words_path, index=False)
    meter.last_tokens.to_csv(tokens_path, index=False)
    seconds = round(time.time() - started)
    new_row = {"text": name, "measured words": len(words), "tokens": len(meter.last_tokens), "seconds": seconds}
    run_log = pd.concat([run_log[run_log["text"] != name], pd.DataFrame([new_row])], ignore_index=True)
    run_log.to_csv(log_path, index=False)
    print(f"{name:34s} {len(words):4d} words  {len(meter.last_tokens):5d} tokens  {seconds:5d} s", flush=True)
run_log

,text,measured words,tokens,seconds
0,creative_gen_detective_switch,263,1000,291
1,creative_gen_detective_t07,256,1000,283
2,creative_gen_detective_t10,241,812,265
3,creative_gen_fairy_switch,243,818,276
4,creative_gen_fairy_t07,229,818,256
5,creative_gen_fairy_t10,291,954,324
6,creative_gen_magic_switch,186,806,211
7,creative_gen_magic_t07,206,825,237
8,creative_gen_magic_t10,264,1000,308


## 5. A second reader
Another model family (`SECOND_READER_MODEL`, OLMo-2 7B Instruct) reads the same texts, plus the clean originals of
the planted texts. Only the single pass over each text is needed (surprisal and the other confidence values of
every token: `tokens_only=True`), so each text takes seconds. Qwen is unloaded first: both don't fit in 8 GB.

In [6]:
del meter, model
torch.cuda.empty_cache()
second_tokenizer, second_model = load_model(SECOND_READER_MODEL, load_in_4bit=LOAD_IN_4BIT)
second_meter = LCAMeter(model=second_model, tokenizer=second_tokenizer)
SECOND_READER_TOKENS_DIR.mkdir(parents=True, exist_ok=True)

second_texts = dict(texts_to_read)
for _name, (original_name, _text, _edits) in planted.items():
    second_texts[original_name] = (TEXTS_DIR / f"{original_name}.txt").read_text(encoding="utf-8")
for name, text in second_texts.items():
    path = SECOND_READER_TOKENS_DIR / f"{name}.csv"
    if path.exists():
        continue
    started = time.time()
    second_meter.run(text, tokens_only=True)
    second_meter.last_tokens.to_csv(path, index=False)
    print(f"{name:34s} {len(second_meter.last_tokens):5d} tokens  {time.time() - started:5.0f} s", flush=True)

system check:
Python path: C:\Users\yoav7\miniconda3\python.exe
Torch version: 2.6.0+cu124
GPU: NVIDIA GeForce RTX 4060
PyTorch: 2.6.0+cu124
CUDA: 12.4
CUDA available: True
Capability: (8, 9)
this is include the following functions torch , torch.nn.functional as F, transformers.AutoTokenizer, transformers.AutoModelForCausalLM


Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

creative_gut_grimm_planted           827 tokens     69 s


creative_gut_oz_planted              814 tokens     75 s


creative_gut_sherlock_planted        812 tokens     59 s


creative_hp3_planted                 918 tokens     35 s


... (17 outputs left out)

creative_hp6                         921 tokens     45 s


factual_wiki_copper                  912 tokens     45 s


factual_wiki_vaccine                 874 tokens     37 s


factual_wiki_volcano                 933 tokens     40 s
